# 05 Models — Logistic Regression — Women's

A linear model to add structural diversity to the ensemble. Tree models and neural nets are all nonlinear and highly correlated. Logistic regression makes fundamentally different errors, which benefits ensemble diversity.

**Inputs** (from S3 `04_preprocessing/womens/`):
- `train_features.parquet`, `stage1_features.parquet`, `stage2_features.parquet`, `feature_columns.parquet`

**Outputs** (to S3 `05_models/logistic_regression/womens/`):
- `oof_predictions.parquet`, `stage1_predictions.parquet`, `stage2_predictions.parquet`, `cv_results.parquet`

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.isotonic import IsotonicRegression

try:
    import optuna
except:
    !pip install optuna
    import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"Optuna version: {optuna.__version__}")

Optuna version: 4.8.0


#### Functions

In [2]:
def read_parquet(filename):
    """Read parquet from S3 or local."""
    try:
        return pd.read_parquet(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_parquet(f"../../04_preprocessing/output/{filename}")

#### Constants

In [3]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "womens"
STAGE = "05_models/logistic_regression"
INPUT_PREFIX = f"s3://{BUCKET}/04_preprocessing/{GENDER}/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

LOCAL_OUTPUT = "output/"

#### Make output directory

In [4]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Data

In [5]:
train = read_parquet("train_features.parquet")
stage1 = read_parquet("stage1_features.parquet")
stage2 = read_parquet("stage2_features.parquet")
feature_cols = read_parquet("feature_columns.parquet")['feature'].tolist()

print(f"Training data: {train.shape}")
print(f"Stage 1 grid: {stage1.shape}")
print(f"Stage 2 grid: {stage2.shape}")
print(f"Features: {len(feature_cols)}")

Training data: (3434, 188)
Stage 1 grid: (258131, 186)
Stage 2 grid: (65703, 185)
Features: 63


## 2. Optuna Hyperparameter Tuning

Use 4-fold Stage 1 cross-validation (2022-2025) with 50 trials to find optimal Logistic Regression hyperparameters.

In [6]:
train_original = train[train['TeamA'] < train['TeamB']].copy()
stage1_folds = [2022, 2023, 2024, 2025]

def make_lr_model(params):
    """Build a LogisticRegression model from Optuna params dict."""
    kwargs = {
        'C': params['C'],
        'max_iter': 2000,
        'random_state': 42,
    }
    penalty = params['penalty']
    kwargs['penalty'] = penalty
    
    if penalty == 'l1':
        kwargs['solver'] = 'saga'
    elif penalty == 'elasticnet':
        kwargs['solver'] = 'saga'
        kwargs['l1_ratio'] = params['l1_ratio']
    else:  # l2
        kwargs['solver'] = 'lbfgs'
    
    return LogisticRegression(**kwargs)

def lr_objective(trial):
    C = trial.suggest_float('C', 0.001, 100.0, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet'])
    
    params = {'C': C, 'penalty': penalty}
    if penalty == 'elasticnet':
        params['l1_ratio'] = trial.suggest_float('l1_ratio', 0.0, 1.0)
    
    fold_scores = []
    for fold in stage1_folds:
        train_mask = train['Fold'] != fold
        val_mask = train_original['Fold'] == fold
        
        X_tr = train.loc[train_mask, feature_cols].fillna(0).values
        y_tr = train.loc[train_mask, 'Label'].values
        X_va = train_original.loc[val_mask, feature_cols].fillna(0).values
        y_va = train_original.loc[val_mask, 'Label'].values
        
        if len(X_va) == 0:
            continue
        
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_va = scaler.transform(X_va)
        
        model = make_lr_model(params)
        model.fit(X_tr, y_tr)
        
        preds = model.predict_proba(X_va)[:, 1]
        fold_scores.append(brier_score_loss(y_va, preds))
    
    return np.mean(fold_scores)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(lr_objective, n_trials=50, show_progress_bar=True)

print(f"Best Stage 1 Brier: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

  0%|          | 0/50 [00:00<?, ?it/s]

Best Stage 1 Brier: 0.1404
Best params: {'C': 0.014906225461039075, 'penalty': 'elasticnet', 'l1_ratio': 0.6236405760010973}


## 3. Set Tuned Logistic Regression Parameters

In [7]:
best = study.best_params

# best = {
#     'C': 0.014906225461039075,
#     'penalty': 'elasticnet',
#     'l1_ratio': 0.6236405760010973,
# }

In [8]:
print("Tuned Logistic Regression parameters:")
for k, v in best.items():
    print(f"  {k}: {v}")

Tuned Logistic Regression parameters:
  C: 0.014906225461039075
  penalty: elasticnet
  l1_ratio: 0.6236405760010973


## 4. LOGO Cross-Validation (with Tuned Params)

In [9]:
folds = sorted(train['Fold'].unique())

oof_preds = []
cv_results = []

for fold in folds:
    train_mask = train['Fold'] != fold
    val_mask = (train_original['Fold'] == fold)
    
    X_train = train.loc[train_mask, feature_cols].fillna(0).values
    y_train = train.loc[train_mask, 'Label'].values
    X_val = train_original.loc[val_mask, feature_cols].fillna(0).values
    y_val = train_original.loc[val_mask, 'Label'].values
    
    if len(X_val) == 0:
        continue
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    
    model = make_lr_model(best)
    model.fit(X_train, y_train)
    
    preds = model.predict_proba(X_val)[:, 1]
    brier = brier_score_loss(y_val, preds)
    
    cv_results.append({
        'Fold': fold,
        'BrierScore': brier,
        'Games': len(y_val),
        'BestRound': 1
    })
    
    fold_oof = train_original.loc[val_mask, ['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val']].copy()
    fold_oof['Pred'] = preds
    oof_preds.append(fold_oof)
    
    print(f"  Fold {fold}: Brier={brier:.4f}, Games={len(y_val)}")

oof_df = pd.concat(oof_preds, ignore_index=True)
cv_df = pd.DataFrame(cv_results)

  Fold 1998: Brier=0.1669, Games=63
  Fold 1999: Brier=0.1416, Games=63
  Fold 2000: Brier=0.1394, Games=63
  Fold 2001: Brier=0.1632, Games=63
  Fold 2002: Brier=0.1296, Games=63
  Fold 2003: Brier=0.1159, Games=63
  Fold 2004: Brier=0.1583, Games=63
  Fold 2005: Brier=0.1563, Games=63
  Fold 2006: Brier=0.1329, Games=63
  Fold 2007: Brier=0.1639, Games=63
  Fold 2008: Brier=0.1159, Games=63
  Fold 2009: Brier=0.1585, Games=63
  Fold 2010: Brier=0.1456, Games=63
  Fold 2011: Brier=0.1307, Games=63
  Fold 2012: Brier=0.1096, Games=63
  Fold 2013: Brier=0.1491, Games=63
  Fold 2014: Brier=0.1335, Games=63
  Fold 2015: Brier=0.1183, Games=63
  Fold 2016: Brier=0.1644, Games=63
  Fold 2017: Brier=0.1351, Games=63
  Fold 2018: Brier=0.1503, Games=63
  Fold 2019: Brier=0.1244, Games=63
  Fold 2021: Brier=0.1545, Games=63
  Fold 2022: Brier=0.1528, Games=67
  Fold 2023: Brier=0.1739, Games=67
  Fold 2024: Brier=0.1215, Games=67
  Fold 2025: Brier=0.1136, Games=67


In [10]:
overall_brier = brier_score_loss(oof_df['Label'], oof_df['Pred'])
stage1_oof = oof_df[oof_df['IsStage1Val'] == 1]
stage1_brier = brier_score_loss(stage1_oof['Label'], stage1_oof['Pred']) if len(stage1_oof) > 0 else None

print(f"\nOverall OOF Brier Score: {overall_brier:.4f}")
print(f"Stage 1 (2022-2025) Brier Score: {stage1_brier:.4f}" if stage1_brier else "No Stage 1 data")
weighted_brier = np.average(cv_df.sort_values('Fold')['BrierScore'], weights=np.arange(1, len(cv_df) + 1))
print(f"Weighted Mean Brier: {weighted_brier:.4f}")


Overall OOF Brier Score: 0.1415
Stage 1 (2022-2025) Brier Score: 0.1404
Weighted Mean Brier: 0.1399


## 5. Train Final Model and Calibrate

In [11]:
X_all = train[feature_cols].fillna(0).values
y_all = train['Label'].values

final_scaler = StandardScaler()
X_all_scaled = final_scaler.fit_transform(X_all)

final_model = make_lr_model(best)
final_model.fit(X_all_scaled, y_all)

calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(oof_df['Pred'].values, oof_df['Label'].values)

oof_df['PredCalibrated'] = calibrator.predict(oof_df['Pred'].values)
calibrated_brier = brier_score_loss(oof_df['Label'], oof_df['PredCalibrated'])
print(f"OOF Brier (raw): {overall_brier:.4f}")
print(f"OOF Brier (calibrated): {calibrated_brier:.4f}")

OOF Brier (raw): 0.1415
OOF Brier (calibrated): 0.1379


## 6. Generate Predictions

In [12]:
X_s1 = final_scaler.transform(stage1[feature_cols].fillna(0).values)
stage1['Pred_logistic_regression'] = final_model.predict_proba(X_s1)[:, 1]
stage1['Pred_logistic_regression_calibrated'] = calibrator.predict(
    stage1['Pred_logistic_regression'].values
).clip(0.02, 0.98)

X_s2 = final_scaler.transform(stage2[feature_cols].fillna(0).values)
stage2['Pred_logistic_regression'] = final_model.predict_proba(X_s2)[:, 1]
stage2['Pred_logistic_regression_calibrated'] = calibrator.predict(
    stage2['Pred_logistic_regression'].values
).clip(0.02, 0.98)

print(f"Stage 1 predictions range: [{stage1['Pred_logistic_regression_calibrated'].min():.3f}, {stage1['Pred_logistic_regression_calibrated'].max():.3f}]")
print(f"Stage 2 predictions range: [{stage2['Pred_logistic_regression_calibrated'].min():.3f}, {stage2['Pred_logistic_regression_calibrated'].max():.3f}]")

stage1_actual = stage1.dropna(subset=['Label'])
if len(stage1_actual) > 0:
    s1_brier = brier_score_loss(stage1_actual['Label'], stage1_actual['Pred_logistic_regression_calibrated'])
    print(f"Stage 1 Brier (calibrated): {s1_brier:.4f}")

Stage 1 predictions range: [0.020, 0.980]
Stage 2 predictions range: [0.020, 0.980]
Stage 1 Brier (calibrated): 0.1346


## 7. Feature Coefficients

In [13]:
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': final_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print("Feature Coefficients (sorted by absolute value):")
for _, row in coef_df.iterrows():
    print(f"  {row['Feature']:30s}: {row['Coefficient']:+.4f}")

Feature Coefficients (sorted by absolute value):
  SeedDiff                      : -0.5093
  SOSDiff                       : +0.3975
  EloDiff                       : +0.3712
  AvgPointDiffDiff              : +0.3626
  SyntheticRankDiff             : -0.2604
  IsPowerConfDiff               : +0.1522
  SeedB                         : +0.1032
  SeedA                         : -0.1032
  PowerConfWinPctDiff           : +0.0593
  Top5WinPctDiff                : +0.0574
  LossStreakDiff                : +0.0571
  FGM_MarginDiff                : +0.0527
  Blk_MarginDiff                : +0.0506
  AvgAstDiff                    : +0.0051
  DefEffDiff                    : +0.0000
  AvgBlkDiff                    : +0.0000
  AvgStlDiff                    : +0.0000
  AvgTODiff                     : +0.0000
  FTPctDiff                     : +0.0000
  OppFGPctDiff                  : +0.0000
  OppFG3PctDiff                 : +0.0000
  AvgDRDiff                     : +0.0000
  AvgORDiff                

## 8. Save Outputs

In [14]:
outputs = {
    'oof_predictions': oof_df[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val', 'Pred', 'PredCalibrated']],
    'stage1_predictions': stage1[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Pred_logistic_regression', 'Pred_logistic_regression_calibrated']],
    'stage2_predictions': stage2[['ID', 'Season', 'TeamA', 'TeamB', 'Pred_logistic_regression', 'Pred_logistic_regression_calibrated']],
    'cv_results': cv_df,
}

for name, df in outputs.items():
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/05_models/logistic_regression/womens/oof_predictions.parquet ((1717, 9))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/logistic_regression/womens/stage1_predictions.parquet ((258131, 7))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/logistic_regression/womens/stage2_predictions.parquet ((65703, 6))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/logistic_regression/womens/cv_results.parquet ((27, 4))


## 9. Summary

In [15]:
print("=" * 60)
print("LOGISTIC REGRESSION MODEL SUMMARY — WOMEN'S")
print("=" * 60)
print(f"\nOOF Brier Score (raw): {overall_brier:.4f}")
print(f"OOF Brier Score (calibrated): {calibrated_brier:.4f}")
weighted_brier = np.average(cv_df.sort_values('Fold')['BrierScore'], weights=np.arange(1, len(cv_df) + 1))
print(f"Weighted Mean Brier: {weighted_brier:.4f}")
print(f"\nTuned hyperparameters:")
for k, v in best.items():
    print(f"  {k}: {v}")
print(f"\nTop 5 features (by |coefficient|):")
for _, row in coef_df.head().iterrows():
    print(f"  {row['Feature']}: {row['Coefficient']:+.4f}")

LOGISTIC REGRESSION MODEL SUMMARY — WOMEN'S

OOF Brier Score (raw): 0.1415
OOF Brier Score (calibrated): 0.1379
Weighted Mean Brier: 0.1399

Tuned hyperparameters:
  C: 0.014906225461039075
  penalty: elasticnet
  l1_ratio: 0.6236405760010973

Top 5 features (by |coefficient|):
  SeedDiff: -0.5093
  SOSDiff: +0.3975
  EloDiff: +0.3712
  AvgPointDiffDiff: +0.3626
  SyntheticRankDiff: -0.2604
